# YOLOv8x + ByteTrack on MOT17
**Run cells top-to-bottom. GPU runtime required (T4 or better).**

| Step | Cell | What it does |
|------|------|--------------|
| 0 | Cell 0 | Fix numpy — run once, then **restart runtime** |
| 1 | Cell 1 | Install all dependencies |
| 2 | Cell 2 | Download MOT17 from Kaggle |
| 3 | Cell 3 | YOLOv8x inference → detection files |
| 4 | Cell 4 | ByteTrack association → track files |
| 5 | Cell 5 | TrackEval evaluation → HOTA/MOTA/IDF1 |

In [ ]:
# ── Cell 0 (new): Reset environment cleanly ───────────────────────────────
# This replaces the old numpy-downgrade approach entirely.
import subprocess, sys

# Remove everything that might conflict
subprocess.run([sys.executable, '-m', 'pip', 'uninstall', '-y',
                'numpy', 'numba', 'boxmot', 'ultralytics', 'lap'],
               capture_output=True)

# Let numpy install at whatever version torch needs (2.x is fine)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'numpy'],
               check=True)

print('Done. NOW: Runtime → Restart session')
print('After restart: skip Cell 0, run from Cell 1.')

Done. NOW: Runtime → Restart session
After restart: skip Cell 0, run from Cell 1.


In [ ]:
# ── Cell 1 (new): Install all dependencies ────────────────────────────────
import subprocess, sys, os

def pip(*args):
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', *args], check=True)

# Install everything — let pip resolve numpy version naturally
pip('ultralytics')
pip('lapx')          # drop-in replacement for lap, numpy 2.x compatible
pip('scipy', 'matplotlib', 'kaggle', 'tqdm')

# Clone and patch TrackEval
if not os.path.exists('/content/TrackEval'):
    subprocess.run(['git', 'clone', '-q',
                    'https://github.com/JonathonLuiten/TrackEval.git',
                    '/content/TrackEval'], check=True)

from pathlib import Path
# Replace the fixes dict in Cell 1 with this:
fixes = {
    'np.float)':  'np.float64)',
    'np.float,':  'np.float64,',
    'np.int)':    'np.int64)',
    'np.int,':    'np.int64,',
    'np.bool)':   'np.bool_)',
    'np.bool,':   'np.bool_,',
}
patched = []
for f in Path('/content/TrackEval').rglob('*.py'):
    txt = f.read_text()
    new = txt
    for old, repl in fixes.items():
        new = new.replace(old, repl)
    if new != txt:
        f.write_text(new); patched.append(f.name)
print(f'TrackEval patched: {len(patched)} files')
import re
for f in Path('/content/TrackEval').rglob('*.py'):
    txt = f.read_text()
    # Fix any .astype(np.bool_) that lost its closing paren
    new = re.sub(r'\.astype\(np\.bool_\s*\n', '.astype(np.bool_)\n', txt)
    # Remove any erroneous () after bool_
    new = re.sub(r'np\.bool_\(\)', 'np.bool_', new)
    if new != txt:
        f.write_text(new)
print('TrackEval patch verified ✓')

# Verify
import numpy as np
import torch
from ultralytics import YOLO
print(f'numpy:      {np.__version__}')
print(f'torch:      {torch.__version__}')
print(f'CUDA:       {torch.cuda.is_available()}')
print('ultralytics: ✓')
print('\nAll dependencies ready ✓')

TrackEval patched: 0 files
TrackEval patch verified ✓
numpy:      2.4.6
torch:      2.11.0+cu128
CUDA:       True
ultralytics: ✓

All dependencies ready ✓


In [ ]:
# ── Cell 2: Download MOT17 from Kaggle ───────────────────────────────────
import os, json, subprocess
from pathlib import Path

KAGGLE_USERNAME = ''   # ← fill in
KAGGLE_KEY      = ''   # ← fill in

os.makedirs('/root/.kaggle', exist_ok=True)
with open('/root/.kaggle/kaggle.json', 'w') as f:
    json.dump({'username': KAGGLE_USERNAME, 'key': KAGGLE_KEY}, f)
os.chmod('/root/.kaggle/kaggle.json', 0o600)

MOT17_ROOT = Path('/content/MOT17/train')

if MOT17_ROOT.exists() and any(MOT17_ROOT.iterdir()):
    print('MOT17 already downloaded, skipping.')
else:
    print('Downloading MOT17 from Kaggle...')
    subprocess.run([
        'kaggle', 'datasets', 'download',
        '-d', 'wenhoujinjust/mot-17',
        '-p', '/content/',
        '--unzip'
    ], check=True)
    print('Done.')

seqs = sorted(p.name for p in MOT17_ROOT.iterdir() if p.is_dir())
print(f'Found {len(seqs)} sequences, e.g.: {seqs[:3]}')

MOT17 already downloaded, skipping.
Found 21 sequences, e.g.: ['MOT17-02-DPM', 'MOT17-02-FRCNN', 'MOT17-02-SDP']


In [ ]:
# ── Cell 3: YOLOv8x inference ─────────────────────────────────────────────
import time, configparser
from pathlib import Path
from tqdm import tqdm
from ultralytics import YOLO
import torch

MOT17_ROOT   = Path('/content/MOT17/train')
DET_OUT_ROOT = Path('/content/detections')
DEVICE       = 'cuda' if torch.cuda.is_available() else 'cpu'
SEQUENCES    = ['MOT17-02-SDP', 'MOT17-04-SDP', 'MOT17-05-SDP',
                'MOT17-09-SDP', 'MOT17-13-SDP']
CONF_THRESH  = 0.35
BATCH_SIZE   = 8

print('Loading YOLOv8x...')
model = YOLO('yolov8x.pt')
print(f'YOLOv8x loaded on {DEVICE} ✓')

def get_seq_info(seq_path):
    cfg = configparser.ConfigParser()
    cfg.read(seq_path / 'seqinfo.ini')
    s = cfg['Sequence']
    return int(s['seqLength']), s.get('imDir', 'img1'), s.get('imExt', '.jpg')

DET_OUT_ROOT.mkdir(parents=True, exist_ok=True)

for seq_name in SEQUENCES:
    seq_path = MOT17_ROOT / seq_name
    seq_len, im_dir, im_ext = get_seq_info(seq_path)
    frames = sorted((seq_path / im_dir).glob(f'*{im_ext}'))
    print(f'\n{seq_name}: {len(frames)} frames')
    lines = []
    t0 = time.time()

    for bs in tqdm(range(0, len(frames), BATCH_SIZE), desc=seq_name):
        batch_frames = frames[bs:bs+BATCH_SIZE]
        results = model(
            [str(f) for f in batch_frames],
            classes=[0], conf=CONF_THRESH,
            verbose=False, device=DEVICE,
        )
        for frame_path, result in zip(batch_frames, results):
            fid = int(frame_path.stem)
            if result.boxes is None or len(result.boxes) == 0:
                continue
            for box in result.boxes:
                x1, y1, x2, y2 = box.xyxy[0].tolist()
                conf = box.conf.item()
                lines.append(
                    f'{fid},-1,{x1:.2f},{y1:.2f},'
                    f'{x2-x1:.2f},{y2-y1:.2f},{conf:.4f},-1,-1,-1')

    fps = len(frames) / (time.time() - t0)
    out = DET_OUT_ROOT / f'{seq_name}.txt'
    out.write_text('\n'.join(lines))
    print(f'  {len(lines)} detections | {fps:.1f} FPS ✓')

print('\nInference complete.')

Loading YOLOv8x...
YOLOv8x loaded on cuda ✓

MOT17-02-SDP: 600 frames


MOT17-02-SDP: 100%|██████████| 75/75 [00:54<00:00,  1.37it/s]


  6754 detections | 10.9 FPS ✓

MOT17-04-SDP: 1050 frames


MOT17-04-SDP: 100%|██████████| 132/132 [01:29<00:00,  1.47it/s]


  16901 detections | 11.7 FPS ✓

MOT17-05-SDP: 837 frames


...

  5793 detections | 14.0 FPS ✓

MOT17-09-SDP: 525 frames


MOT17-09-SDP: 100%|██████████| 66/66 [00:43<00:00,  1.53it/s]


  5021 detections | 12.2 FPS ✓

MOT17-13-SDP: 750 frames


MOT17-13-SDP: 100%|██████████| 94/94 [01:02<00:00,  1.50it/s]

  5396 detections | 12.0 FPS ✓

Inference complete.


In [ ]:
# ── Cell 4: ByteTrack ─────────────────────────────────
import numpy as np
import configparser
from pathlib import Path
from tqdm import tqdm
from scipy.optimize import linear_sum_assignment

# Hungarian assignment using scipy
def linear_assignment(cost, thresh):
    if cost.size == 0:
        return [], list(range(cost.shape[0])), list(range(cost.shape[1]))
    cost_w = cost.copy()
    cost_w[cost_w > thresh] = thresh + 1e-4
    row_ind, col_ind = linear_sum_assignment(cost_w)
    matches, matched_r, matched_c = [], set(), set()
    for r, c in zip(row_ind, col_ind):
        if cost[r, c] <= thresh:
            matches.append([r, c])
            matched_r.add(r); matched_c.add(c)
    u_rows = [r for r in range(cost.shape[0]) if r not in matched_r]
    u_cols = [c for c in range(cost.shape[1]) if c not in matched_c]
    return matches, u_rows, u_cols

# Kalman Filter
class KalmanFilter:
    def __init__(self):
        ndim, dt = 4, 1.
        self._motion_mat = np.eye(2*ndim, 2*ndim)
        for i in range(ndim): self._motion_mat[i, ndim+i] = dt
        self._update_mat = np.eye(ndim, 2*ndim)
        self._std_weight_position = 1./20
        self._std_weight_velocity = 1./160

    def initiate(self, m):
        mean = np.r_[m, np.zeros_like(m)]
        std  = [2*self._std_weight_position*m[3],
                2*self._std_weight_position*m[3],
                1e-2,
                2*self._std_weight_position*m[3],
                10*self._std_weight_velocity*m[3],
                10*self._std_weight_velocity*m[3],
                1e-5,
                10*self._std_weight_velocity*m[3]]
        return mean, np.diag(np.square(std))

    def predict(self, mean, cov):
        std = [self._std_weight_position*mean[3],
               self._std_weight_position*mean[3],
               1e-2,
               self._std_weight_position*mean[3],
               self._std_weight_velocity*mean[3],
               self._std_weight_velocity*mean[3],
               1e-5,
               self._std_weight_velocity*mean[3]]
        Q    = np.diag(np.square(std))
        mean = self._motion_mat @ mean
        cov  = self._motion_mat @ cov @ self._motion_mat.T + Q
        return mean, cov

    def update(self, mean, cov, measurement):
        std = [self._std_weight_position*mean[3],
               self._std_weight_position*mean[3],
               1e-1,
               self._std_weight_position*mean[3]]
        R = np.diag(np.square(std))
        S = self._update_mat @ cov @ self._update_mat.T + R
        K = cov @ self._update_mat.T @ np.linalg.inv(S)
        mean = mean + K @ (measurement - self._update_mat @ mean)
        cov  = cov - K @ S @ K.T
        return mean, cov

# Track state and STrack
class TrackState:
    New=0; Tracked=1; Lost=2; Removed=3

class STrack:
    _count = 0
    def __init__(self, tlwh, score):
        self._tlwh=np.array(tlwh,dtype=float); self.score=score
        self.is_activated=False; self.state=TrackState.New
        self.mean=None; self.covariance=None; self.tracklet_len=0
        self.frame_id=0; self.start_frame=0; self.kalman_filter=None
        STrack._count+=1; self.track_id=STrack._count

    @staticmethod
    def tlwh_to_xyah(tlwh):
        ret=np.array(tlwh,dtype=float)
        ret[:2]+=ret[2:]/2; ret[2]/=ret[3]; return ret

    @property
    def tlwh(self):
        if self.mean is None: return self._tlwh.copy()
        ret=self.mean[:4].copy(); ret[2]*=ret[3]; ret[:2]-=ret[2:]/2
        return ret

    def activate(self, kf, fid):
        self.kalman_filter=kf
        self.mean,self.covariance=kf.initiate(self.tlwh_to_xyah(self._tlwh))
        self.state=TrackState.Tracked; self.is_activated=True
        self.frame_id=fid; self.start_frame=fid; self.tracklet_len=0

    def re_activate(self, new_track, fid):
        self.mean,self.covariance=self.kalman_filter.update(
            self.mean,self.covariance,self.tlwh_to_xyah(new_track._tlwh))
        self.state=TrackState.Tracked; self.is_activated=True
        self.frame_id=fid; self.tracklet_len=0; self.score=new_track.score

    def update(self, new_track, fid):
        self.mean,self.covariance=self.kalman_filter.update(
            self.mean,self.covariance,self.tlwh_to_xyah(new_track._tlwh))
        self.state=TrackState.Tracked; self.is_activated=True
        self.frame_id=fid; self.tracklet_len+=1; self.score=new_track.score

# IoU distance
def iou_distance(atracks, btracks):
    if not atracks or not btracks:
        return np.zeros((len(atracks), len(btracks)))
    def to_xyxy(t): x,y,w,h=t.tlwh; return [x,y,x+w,y+h]
    ab=np.array([to_xyxy(t) for t in atracks])
    bb=np.array([to_xyxy(t) for t in btracks])
    ious=np.zeros((len(ab),len(bb)))
    for i,a in enumerate(ab):
        xi1=np.maximum(a[0],bb[:,0]); yi1=np.maximum(a[1],bb[:,1])
        xi2=np.minimum(a[2],bb[:,2]); yi2=np.minimum(a[3],bb[:,3])
        inter=np.maximum(xi2-xi1,0)*np.maximum(yi2-yi1,0)
        aa=(a[2]-a[0])*(a[3]-a[1])
        ba=(bb[:,2]-bb[:,0])*(bb[:,3]-bb[:,1])
        ious[i]=inter/(aa+ba-inter+1e-6)
    return 1-ious

# ByteTracker
class ByteTracker:
    def __init__(self,track_high_thresh=0.6,track_low_thresh=0.1,
                 new_track_thresh=0.7,track_buffer=30,match_thresh=0.8):
        self.high_thresh=track_high_thresh; self.low_thresh=track_low_thresh
        self.new_thresh=new_track_thresh; self.match_thresh=match_thresh
        self.buffer=track_buffer; self.kf=KalmanFilter()
        self.tracked=[]; self.lost=[]; self.frame_id=0; STrack._count=0

    def update(self, dets):
        self.frame_id+=1; fid=self.frame_id
        scores=dets[:,4] if len(dets) else np.array([])
        def make(d):
            return [STrack([r[0],r[1],r[2]-r[0],r[3]-r[1]],r[4]) for r in d]
        high=make(dets[scores>=self.high_thresh]) if len(dets) else []
        low =make(dets[(scores>=self.low_thresh)&(scores<self.high_thresh)]) if len(dets) else []
        for t in self.tracked+self.lost:
            t.mean,t.covariance=self.kf.predict(t.mean,t.covariance)
        active=[t for t in self.tracked if t.is_activated]
        inactive=[t for t in self.tracked if not t.is_activated]
        pool=active+inactive
        m1,u_trk1,u_det1=linear_assignment(iou_distance(pool,high),self.match_thresh)
        for ti,di in m1: pool[ti].update(high[di],fid)
        u_active=[active[i] for i in u_trk1 if i<len(active)]
        m2,u_trk2,_=linear_assignment(iou_distance(u_active,low),0.5)
        for ti,di in m2: u_active[ti].update(low[di],fid)
        newly_lost=[u_active[i] for i in u_trk2]
        for t in newly_lost: t.state=TrackState.Lost
        m3,_,u_det2=linear_assignment(
            iou_distance(self.lost,[high[i] for i in u_det1]),0.5)
        for ti,di in m3: self.lost[ti].re_activate(high[u_det1[di]],fid)
        for i in u_det2:
            d=high[u_det1[i]]
            if d.score>=self.new_thresh:
                d.activate(self.kf,fid); self.tracked.append(d)
        reactivated=[self.lost[ti] for ti,_ in m3]
        self.lost=[t for t in self.lost if t not in reactivated]+newly_lost
        self.lost=[t for t in self.lost if fid-t.frame_id<=self.buffer]
        self.tracked=[t for t in self.tracked+reactivated
                      if t.state==TrackState.Tracked]
        return [t for t in self.tracked if t.is_activated]

print('ByteTracker ready ✓')

# Run tracking
DET_ROOT   = Path('/content/detections')
TRACK_ROOT = Path('/content/tracks')
MOT17_ROOT = Path('/content/MOT17/train')
SEQUENCES  = ['MOT17-02-SDP','MOT17-04-SDP','MOT17-05-SDP',
               'MOT17-09-SDP','MOT17-13-SDP']

def get_fps_and_len(p):
    cfg=configparser.ConfigParser(); cfg.read(p/'seqinfo.ini')
    s=cfg['Sequence']; return float(s.get('frameRate',30)),int(s['seqLength'])

def load_dets(f):
    d={}
    for line in f.read_text().strip().splitlines():
        if not line.strip(): continue
        p=line.split(',')
        fid=int(p[0]); x1,y1,w,h,c=float(p[2]),float(p[3]),float(p[4]),float(p[5]),float(p[6])
        d.setdefault(fid,[]).append([x1,y1,x1+w,y1+h,c])
    return {k:np.array(v,dtype=np.float32) for k,v in d.items()}

TRACK_ROOT.mkdir(parents=True,exist_ok=True)

for seq_name in SEQUENCES:
    fps,n_frames=get_fps_and_len(MOT17_ROOT/seq_name)
    dets=load_dets(DET_ROOT/f'{seq_name}.txt')
    tracker=ByteTracker(track_high_thresh=0.6,track_low_thresh=0.1,
                        new_track_thresh=0.7,track_buffer=int(fps),match_thresh=0.8)
    lines=[]
    for fid in tqdm(range(1,n_frames+1),desc=seq_name):
        fd=dets.get(fid,np.empty((0,5),dtype=np.float32))
        for t in tracker.update(fd):
            x,y,w,h=t.tlwh
            lines.append(f'{fid},{t.track_id},{x:.2f},{y:.2f},{w:.2f},{h:.2f},{t.score:.4f},-1,-1,-1')
    (TRACK_ROOT/f'{seq_name}.txt').write_text('\n'.join(lines))
    ids=len({int(l.split(',')[1]) for l in lines})
    print(f'  {seq_name}: {ids} unique IDs, {len(lines)} track rows')

print('\nTracking complete.')

ByteTracker ready ✓


MOT17-02-SDP: 100%|██████████| 600/600 [00:00<00:00, 714.67it/s]


  MOT17-02-SDP: 49 unique IDs, 4042 track rows


MOT17-04-SDP: 100%|██████████| 1050/1050 [00:04<00:00, 219.74it/s]


  MOT17-04-SDP: 49 unique IDs, 12316 track rows


...

  MOT17-05-SDP: 169 unique IDs, 4759 track rows


MOT17-09-SDP: 100%|██████████| 525/525 [00:01<00:00, 327.68it/s]


  MOT17-09-SDP: 63 unique IDs, 4096 track rows


MOT17-13-SDP: 100%|██████████| 750/750 [00:01<00:00, 449.45it/s]


  MOT17-13-SDP: 74 unique IDs, 2923 track rows

Tracking complete.


In [ ]:
# ── Cell 5: TrackEval evaluation → HOTA / MOTA / IDF1 ────────────────────
import os, sys, shutil, json
import numpy as np
from pathlib import Path

sys.path.insert(0, '/content/TrackEval')
import trackeval

TRACKEVAL_DIR = Path('/content/TrackEval')
MOT17_ROOT    = Path('/content/MOT17/train')
TRACK_ROOT    = Path('/content/tracks')
RESULTS_DIR   = Path('/content/trackeval_results')
TRACKER_NAME  = 'YOLOv8x-ByteTrack'
SEQUENCES     = ['MOT17-02-SDP','MOT17-04-SDP','MOT17-05-SDP','MOT17-09-SDP','MOT17-13-SDP']

GT_DIR      = TRACKEVAL_DIR/'data/gt/mot_challenge/MOT17-train'
TRACKER_DIR = TRACKEVAL_DIR/'data/trackers/mot_challenge'/'MOT17-train'/TRACKER_NAME/'data'

# Rebuild directories cleanly
for d in [GT_DIR, TRACKER_DIR, RESULTS_DIR]:
    if d.exists(): shutil.rmtree(d)
    d.mkdir(parents=True)

print('Copying GT and tracker files:')
for seq in SEQUENCES:
    (GT_DIR/seq/'gt').mkdir(parents=True)
    (GT_DIR/seq/'seqinfo.ini').write_text((MOT17_ROOT/seq/'seqinfo.ini').read_text())
    (GT_DIR/seq/'gt'/'gt.txt').write_bytes((MOT17_ROOT/seq/'gt'/'gt.txt').read_bytes())
    (TRACKER_DIR/f'{seq}.txt').write_text((TRACK_ROOT/f'{seq}.txt').read_text())
    print(f'  {seq} ✓')

print('\nRunning TrackEval...')
eval_config = trackeval.Evaluator.get_default_eval_config()
eval_config.update({'USE_PARALLEL':False,'PRINT_RESULTS':True,'PRINT_ONLY_COMBINED':False,
                    'TIME_PROGRESS':True,'OUTPUT_SUMMARY':True,'OUTPUT_DETAILED':True,
                    'PLOT_CURVES':False,'OUTPUT_FOLDER':str(RESULTS_DIR)})

dataset_config = trackeval.datasets.MotChallenge2DBox.get_default_dataset_config()
dataset_config.update({
    'GT_FOLDER':          str(TRACKEVAL_DIR/'data/gt/mot_challenge'),
    'TRACKERS_FOLDER':    str(TRACKEVAL_DIR/'data/trackers/mot_challenge'),
    'BENCHMARK':          'MOT17',
    'SPLIT_TO_EVAL':      'train',
    'TRACKERS_TO_EVAL':   [TRACKER_NAME],
    'CLASSES_TO_EVAL':    ['pedestrian'],
    'TRACKER_SUB_FOLDER': 'data',
    'SKIP_SPLIT_FOL':     False,
    'PLOT_CURVES':        False,
    'SEQ_INFO': {'MOT17-02-SDP':600,'MOT17-04-SDP':1050,'MOT17-05-SDP':837,
                 'MOT17-09-SDP':525,'MOT17-13-SDP':750},
})

evaluator    = trackeval.Evaluator(eval_config)
dataset_list = [trackeval.datasets.MotChallenge2DBox(dataset_config)]
results, _   = evaluator.evaluate(dataset_list, [
    trackeval.metrics.HOTA(),
    trackeval.metrics.CLEAR(),
    trackeval.metrics.Identity(),
])

# Display results
METRICS = {
    'HOTA':('HOTA','HOTA(0)'), 'DetA':('HOTA','DetA'), 'AssA':('HOTA','AssA'),
    'MOTA':('CLEAR','MOTA'),   'IDF1':('Identity','IDF1'),
    'Recall':('CLEAR','CLR_Re'), 'Precision':('CLEAR','CLR_Pr'),
}

def get_val(seq_res, sub, key):
    v = seq_res[sub][key]
    v = float(v.mean()) if isinstance(v, np.ndarray) else float(v)
    return v*100 if -1.0<=v<=1.0 else v

tracker_res  = results['MotChallenge2DBox'][TRACKER_NAME]
main_metrics = ['HOTA','DetA','AssA','MOTA','IDF1','Recall','Precision']

print('\n'+'='*80)
print('RESULTS: YOLOv8x + ByteTrack on MOT17 (5-sequence subset)')
print('='*80)
print(f"{'Sequence':<25}" + ''.join(f'{m:>11}' for m in main_metrics))
print('─'*(25+11*len(main_metrics)))

for seq in SEQUENCES+['COMBINED_SEQ']:
    if seq not in tracker_res: continue
    label   = 'COMBINED' if seq=='COMBINED_SEQ' else seq
    seq_res = tracker_res[seq]['pedestrian']
    row     = f'{label:<25}'
    for m in main_metrics:
        sub,key=METRICS[m]
        try: row+=f'{get_val(seq_res,sub,key):>10.1f}%'
        except: row+=f'{"N/A":>11}'
    print(row)

print('─'*(25+11*len(main_metrics)))
print()
print('Reference (ByteTrack paper, MOT17 train, YOLOX-X fine-tuned):')
print('  HOTA: 63.1   MOTA: 78.5   IDF1: 73.4')

# Save JSON summary
ALL_METRICS = {**METRICS, 'IDSW':('CLEAR','IDSW'), 'MT':('CLEAR','MT'), 'ML':('CLEAR','ML')}
output={}
for seq in SEQUENCES+['COMBINED_SEQ']:
    if seq not in tracker_res: continue
    seq_res=tracker_res[seq]['pedestrian']
    output[seq]={m:round(get_val(seq_res,sub,key),2) for m,(sub,key) in ALL_METRICS.items()}
RESULTS_DIR.mkdir(exist_ok=True)
(RESULTS_DIR/'summary.json').write_text(json.dumps(output,indent=2))
print(f'\nResults saved to /content/trackeval_results/summary.json')

Copying GT and tracker files:
  MOT17-02-SDP ✓
  MOT17-04-SDP ✓
  MOT17-05-SDP ✓
  MOT17-09-SDP ✓
  MOT17-13-SDP ✓

Running TrackEval...

Eval Config:
USE_PARALLEL         : False                         
NUM_PARALLEL_CORES   : 8                             
BREAK_ON_ERROR       : True                          
RETURN_ON_ERROR      : False                         
LOG_ON_ERROR         : /content/TrackEval/error_log.txt
PRINT_RESULTS        : True                          
PRINT_ONLY_COMBINED  : False                         
PRINT_CONFIG         : True                          
TIME_PROGRESS        : True                          
DISPLAY_LESS_PROGRESS : True                          
OUTPUT_SUMMARY       : True                          
OUTPUT_EMPTY_CLASSES : True                          
OUTPUT_DETAILED      : True                          
PLOT_CURVES          : False                         
OUTPUT_FOLDER        : /content/trackeval_results    

MotChallenge2DBox Config:
GT_FOLDER